In [ ]:
from pathlib import Path
import torch

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")
    REPO = Path("/content/motionnet_cnn")
    if REPO.exists():
        !cd {REPO} && git pull
    else:
        !git clone https://github.com/guardomayas/motionnet.git {REPO}
    !pip install -q -e {REPO}
    import site, importlib
    site.main(); importlib.invalidate_caches()

from motionnet.paths import setup, compose
from motionnet.dataset import get_data, batches 
from torch.utils.data import DataLoader
import seaborn as sns

sns.set_context("talk")
P = setup(IN_COLAB)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

cfg, _ = compose(exp_name="s01_lambda2", data="eye31_150fps", model="base", corpus=P.corpus)
data = get_data(cfg["stimulus"], P.movies, device)
info = data.info

In [ ]:
L = cfg["loss"]
L["lambda_var"] = 1.0        # cfg["loss"]["lambda_var"] is now 1.0 as well
L["lambda_1"] = 0.1
cfg["model"]["input_noise_std"] = 0.42      # 3× the base 0.1414

In [ ]:
train_dl = batches(data.train, 32, shuffle=True, drop_last=True)
val_dl   = list(batches(data.val, 32))

In [ ]:
from motionnet.models import MotionCNN 
import torch.nn as nn
GRAY  = info.gray_frames          # hoisted: the loaders no longer touch train_ds

probe = MotionCNN(
                 rf_size_1=5,
                 rf_size_2=5,
                 fps = info.fps,
                 num_subunits_1=8,
                 num_subunits_2=8,
                 out_gain=1.0,
                 initial_threshold=0.0,
                 initial_gain=1.0,
                 input_noise_std = 0.1,
                 layer_noise_std = 0.1
).to(device)

In [ ]:
from motionnet.train.loop import compute_v_std
loss_fn = nn.MSELoss()
probe_opt = torch.optim.AdamW(probe.parameters(), lr=1e-3, weight_decay=0.0)

batch  = next(iter(train_dl))
movie  = batch["movie"].to(device)

valid = slice(info.gray_frames + probe.max_delay - 1, None)
V_STD = compute_v_std(data.train, valid, device)

print("V_STD:", V_STD.cpu().numpy().round(1))
target = batch["vel_deg_s"].to(device) / V_STD
probe.train()

for step in range(500):
    pred = probe(movie)
    loss = loss_fn(pred[:, valid], target[:, valid])
    probe_opt.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(probe.parameters(), max_norm=1.0)
    probe_opt.step()

    if step % 250 == 0:
        with torch.no_grad():
            t = target[:, valid]
            print(f"{step:4d}  loss {loss.item():.4f}  R² {1 - loss.item()/t.var().item():.3f}")

del probe, probe_opt
torch.cuda.empty_cache()

t = target[:, valid]
print("target var:", t.var().item())                    # predict-mean baseline
print("R²:", 1 - loss.item() / t.var().item())
print("pred std:", pred[:, valid].std().item(), "target std:", t.std().item())
err = (pred[:, valid] - target[:, valid]).pow(2).mean((0, 1))
print(1 - err / target[:, valid].var((0, 1)))


## Module based training loop

In [ ]:
from motionnet.train import fit, compute_v_std

model, history, ckpt = fit(cfg, data, device, P.ckpt, tag="adhoc")

# downstream analysis cells expect these
GRAY   = info.gray_frames
VALID  = slice(GRAY + model.max_delay - 1, None)
V_STD = compute_v_std(data.train, valid, device)
val_dl = list(batches(data.val, cfg["optim"]["batch_size"]))

In [ ]:
import pandas as pd
va = pd.DataFrame(history["val"])
tr = pd.DataFrame(history["train"])
print(va[["mse", "svar"]].iloc[::20])        # every 20th epoch

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
epochs = range(1, len(history["train"]) + 1)


tr = pd.DataFrame(history["train"])
va = pd.DataFrame(history["val"])
epochs = range(1, len(tr) + 1)

fig, ax = plt.subplots(1, 4, figsize=(15, 4))

ax[0].plot(epochs, tr["mse"], label="Train")
ax[0].plot(epochs, va["mse"], label="Validation")
ax[0].axhline(1.0, ls="--", lw=1, c="k", label="Predict zero")
ax[0].set(xlabel="Epoch", ylabel="Velocity MSE (std units)")

ax[1].plot(epochs, va["r2_x"], label="$v_x$")
ax[1].plot(epochs, va["r2_y"], label="$v_y$")
ax[1].set(xlabel="Epoch", ylabel="Validation R²", ylim=(0, 1))

ax[2].plot(epochs, va["rate1"], label="Layer 1")
ax[2].plot(epochs, va["rate2"], label="Layer 2")
ax[2].set(xlabel="Epoch", ylabel="Mean rate")

ax[3].plot(epochs, va["svar"], label="Spatial var")

for a in ax:
    a.legend(frameon=False)
fig.suptitle(f"λ₁={L['lambda_1']:g}, λ₂={L['lambda_2']:g}")
fig.tight_layout()
plt.show()

In [ ]:
import numpy as np
from scipy.signal import convolve2d

def temporal_kernels(cnn):
    """(N1, D) kernels in plot orientation: lag 0 LAST, matching t_ms."""
    phi = cnn.temporal_w @ cnn.tbasis.T                 # (N1, D), lag 0 first
    phi = phi / (phi.norm(dim=1, keepdim=True) + 1e-8)  # same as forward()
    return phi.flip(-1).detach().cpu().numpy()          # lag 0 last

def acceptance_kernel(sigma_taps, truncate=3.0):
    """Ommatidial acceptance function sampled on the tap lattice."""
    r = max(1, int(round(truncate * sigma_taps)))
    ax = np.arange(-r, r + 1)
    g = np.exp(-ax ** 2 / (2 * sigma_taps ** 2))
    k = np.outer(g, g)
    return k / k.sum()


def effective_rf(cnn, dataset, truncate=3.0):
    """W1 convolved with the optics: the object comparable to a measured RF.

    The blur acted upstream of the network, so the learned weights carry no
    optics. Reverse-correlating the model against the *pre-blur* stimulus
    would recover this; convolving is exact because that path is linear.
    """
    sw = cnn.spatial_kernel[:, 0].detach().cpu().clone().numpy()
    sigma_taps = dataset.sigma_px / dataset.spacing_px
    k = acceptance_kernel(sigma_taps, truncate)
    eff = np.stack([convolve2d(w, k, mode="full") for w in sw])
    return eff, sigma_taps

def plot_cnn_subunits_1(cnn, dataset, effective=True, truncate=3.0):
    n_su, max_delay = cnn.num_subunits_1, cnn.max_delay
    tw = temporal_kernels(cnn)

    if effective:
        sw, sigma_taps = effective_rf(cnn, dataset, truncate)
        label = f"W₁ ⊛ O  (σ = {sigma_taps:.2f} taps)"
    else:
        sw = cnn.spatial_kernel[:, 0].detach().cpu().clone().numpy()
        label = "W₁ (weights only)"

    for n in range(n_su):                       # fix sign, push gain to time
        if abs(sw[n].min()) > sw[n].max():
            sw[n], tw[n] = -sw[n], -tw[n]
        s = np.linalg.norm(sw[n])
        sw[n] /= s
        tw[n] *= s

    k = sw.shape[-1]
    half = k * dataset.deg_per_tap / 2
    extent = [-half, half, -half, half]
    t_ms = np.arange(-max_delay + 1, 1) * (1000.0 / dataset.fps)
    vlim, ylim = np.abs(sw).max(), np.abs(tw).max()

    fig, axs = plt.subplots(n_su, 3, figsize=(8, 3.2 * n_su), squeeze=False,
                            gridspec_kw=dict(width_ratios=[1, 1.9, 0.1]))
    for n in range(n_su):
        axs[n, 0].plot(t_ms, tw[n])
        axs[n, 0].axhline(0, ls=":", c="k", lw=0.8)
        axs[n, 0].set_ylim(-ylim, ylim)
        axs[n, 0].tick_params(labelbottom=(n == n_su - 1))
        im = axs[n, 1].imshow(sw[n], cmap="RdBu_r", vmin=-vlim, vmax=vlim,
                              extent=extent, interpolation="nearest")
        axs[n, 1].set_title(f"subunit 1,{n+1}")
        plt.colorbar(im, cax=axs[n, 2])
    axs[-1, 0].set_xlabel(r"$\Delta t$ [ms]")
    axs[-1, 1].set_xlabel("azimuth [deg]")
    fig.suptitle(label)
    fig.tight_layout()
    
    return fig, axs



def plot_cnn_subunits2(cnn: nn.Module):
    """
    Plot the second layer of subunits of a CNN.
    """
    cnn_filters_2 = cnn.layer2.weight.to("cpu").detach()

    fig, axs = plt.subplots(cnn.num_subunits_2,
                            cnn.num_subunits_1,
                            figsize=(4 * cnn.num_subunits_2,
                                    4 * cnn.num_subunits_1),
                            sharex=True, sharey=True)
    # vlim = abs(cnn_filters_2).max()
    percentile = 99
    vlim = torch.quantile(cnn_filters_2.abs().flatten(), percentile / 100).item()
    for i in range(cnn.num_subunits_2):
        for j in range(cnn.num_subunits_1):
            axs[i, j].imshow(cnn_filters_2[i, j],
                            vmin=-vlim, vmax=vlim, cmap="RdBu_r")

            axs[i, j].set_title('subunit 1,{} $\\to$ 2,{}'.format(j+1,i+1))


In [ ]:
plot_cnn_subunits_1(model, info, effective=True)